# Santander Value Prediction — Ensemble Models

Compare Random Forest, Gradient Boosting, and LightGBM with consistent 5-fold CV on the training partition. Models learn on `log1p(target)`; the reference decision tree achieved **1.6458** mean CV RMSLE.

**Selection criterion:** mean CV RMSLE; fitting time and train–validation gap are supporting diagnostics.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    KFold,
    cross_val_score,
    train_test_split,
)

dataset = fetch_openml(
    data_id=42572,
    target_column="target",
    as_frame=True,
)

X = dataset.data
y = dataset.target

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

In [2]:
rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=10,
    min_samples_leaf=50,
    max_features=0.5,
    bootstrap=True,
    random_state=42,
    n_jobs=-1,
)

rf_scores = cross_val_score(
    rf_model,
    X_train,
    np.log1p(y_train),
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=1,
)

rf_rmsle = -rf_scores

print("Fold RMSLE:", rf_rmsle)
print(f"Mean CV RMSLE: {rf_rmsle.mean():.4f}")
print(f"Std CV RMSLE: {rf_rmsle.std():.4f}")

Fold RMSLE: [1.54113376 1.56462223 1.57456051 1.50955674 1.52369455]
Mean CV RMSLE: 1.5427
Std CV RMSLE: 0.0243


**Random Forest reference:** 50 trees achieve **1.5427 ± 0.0243** mean CV RMSLE, better than the selected decision tree (**1.6458**).

## 2. Random Forest: number of trees

Compare 25, 50, and 100 trees to balance error reduction against training time.

In [3]:
from sklearn.model_selection import cross_validate

rf_results = []

for n_trees in [25, 50, 100]:
    model = RandomForestRegressor(
        n_estimators=n_trees,
        max_depth=10,
        min_samples_leaf=50,
        max_features=0.5,
        bootstrap=True,
        random_state=42,
        n_jobs=-1,
    )

    scores = cross_validate(
        model,
        X_train,
        np.log1p(y_train),
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
    )

    rmsle = -scores["test_score"]

    rf_results.append({
        "n_estimators": n_trees,
        "mean_rmsle": rmsle.mean(),
        "std_rmsle": rmsle.std(),
        "mean_fit_time": scores["fit_time"].mean(),
    })

results_df = pd.DataFrame(rf_results)

display(results_df)

,n_estimators,mean_rmsle,std_rmsle,mean_fit_time
0,25,1.546542,0.022690,0.816319
1,50,1.542714,0.024322,1.183261
2,100,1.542053,0.023009,2.265117


**Result:** CV RMSLE improves from **1.5465** (25 trees) to **1.5427** (50) and **1.5421** (100). The additional gain beyond 50 trees is minimal; retain **50 trees** as an efficient reference.

## 3. Gradient Boosting

Test sequential tree fitting against the Random Forest reference.

In [4]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_validate

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=20,
    max_features=0.5,
    random_state=42,
)

gb_scores = cross_validate(
    gb_model,
    X_train,
    np.log1p(y_train),
    cv=cv,
    scoring="neg_root_mean_squared_error",
    return_train_score=True,
    n_jobs=1,
)

gb_train_rmsle = -gb_scores["train_score"]
gb_val_rmsle = -gb_scores["test_score"]

print("Fold RMSLE:", gb_val_rmsle)
print(f"Mean CV RMSLE: {gb_val_rmsle.mean():.4f}")
print(f"Std CV RMSLE: {gb_val_rmsle.std():.4f}")
print(f"Mean Train RMSLE: {gb_train_rmsle.mean():.4f}")
print(f"Mean Fit Time: {gb_scores['fit_time'].mean():.2f}s")

Fold RMSLE: [1.52449524 1.55764672 1.57092036 1.49611216 1.50390571]
Mean CV RMSLE: 1.5306
Std CV RMSLE: 0.0293
Mean Train RMSLE: 1.4157
Mean Fit Time: 7.01s


**Result:** Gradient Boosting reaches **1.5306 ± 0.0293** mean CV RMSLE (training: **1.4157**), improving on Random Forest (**1.5427**) at higher fitting cost (6.87s vs. ~1.18s per fold, with different threading settings).

## 4. Initial LightGBM

Try histogram-based, leaf-wise boosting with a constrained number of leaves on this small, high-dimensional dataset.

In [5]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_validate

lgbm_model = LGBMRegressor(
    n_estimators=150,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=6,
    min_child_samples=40,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1,
    n_jobs=4,
)

lgbm_scores = cross_validate(
    lgbm_model,
    X_train,
    np.log1p(y_train),
    cv=cv,
    scoring="neg_root_mean_squared_error",
    return_train_score=True,
    n_jobs=1,
)

lgbm_train_rmsle = -lgbm_scores["train_score"]
lgbm_val_rmsle = -lgbm_scores["test_score"]

print("Fold RMSLE:", lgbm_val_rmsle)
print(f"Mean CV RMSLE: {lgbm_val_rmsle.mean():.4f}")
print(f"Std CV RMSLE: {lgbm_val_rmsle.std():.4f}")
print(f"Mean Train RMSLE: {lgbm_train_rmsle.mean():.4f}")
print(f"Generalization Gap: {lgbm_val_rmsle.mean() - lgbm_train_rmsle.mean():.4f}")
print(f"Mean Fit Time: {lgbm_scores['fit_time'].mean():.2f}s")

Fold RMSLE: [1.46269191 1.50038382 1.50650113 1.45314501 1.43796101]
Mean CV RMSLE: 1.4721
Std CV RMSLE: 0.0268
Mean Train RMSLE: 1.2121
Generalization Gap: 0.2600
Mean Fit Time: 0.96s


**Result:** LightGBM achieves **1.4721 ± 0.0268** CV RMSLE (training: **1.2121**; gap: **0.2600**) and ~1.01s fit time per fold. It outperforms the earlier ensembles; inspect regularization before selecting it.

## 5. LightGBM: number of leaves

Compare 7, 15, and 31 leaves at fixed settings.

In [6]:
leaf_results = []

for leaves in [7, 15, 31]:

    model = LGBMRegressor(
        n_estimators=150,
        learning_rate=0.05,
        num_leaves=leaves,
        max_depth=6,
        min_child_samples=40,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        random_state=42,
        verbosity=-1,
        n_jobs=4,
    )

    scores = cross_validate(
        model,
        X_train,
        np.log1p(y_train),
        cv=cv,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
        n_jobs=1,
    )

    train_rmsle = -scores["train_score"]
    val_rmsle = -scores["test_score"]

    leaf_results.append({
        "num_leaves": leaves,
        "train_rmsle": train_rmsle.mean(),
        "cv_rmsle": val_rmsle.mean(),
        "std_cv": val_rmsle.std(),
        "generalization_gap": (
            val_rmsle.mean() - train_rmsle.mean()
        ),
        "fit_time": scores["fit_time"].mean(),
    })

leaf_df = pd.DataFrame(leaf_results)

display(leaf_df.round(4))

,num_leaves,train_rmsle,cv_rmsle,std_cv,generalization_gap,fit_time
0,7,1.3138,1.4865,0.0309,0.1726,0.9004
1,15,1.2121,1.4721,0.0268,0.2600,0.8757
2,31,1.1921,1.4703,0.0277,0.2782,0.9135


**Result:** CV RMSLE = **1.4865** (7), **1.4721** (15), **1.4703** (31). The small gain from 15 to 31 comes with a wider train–validation gap. Retain **15** provisionally for a simpler model.

## 6. LightGBM: minimum leaf size

Compare `min_child_samples` = 20, 40, and 80 to limit overly specific leaves.

In [7]:
child_results = []

for min_samples in [20, 40, 80]:
    model = LGBMRegressor(
        n_estimators=150,
        learning_rate=0.05,
        num_leaves=15,
        max_depth=6,
        min_child_samples=min_samples,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        random_state=42,
        verbosity=-1,
        n_jobs=4,
    )

    scores = cross_validate(
        model,
        X_train,
        np.log1p(y_train),
        cv=cv,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
        n_jobs=1,
    )

    train_rmsle = -scores["train_score"]
    val_rmsle = -scores["test_score"]

    child_results.append({
        "min_child_samples": min_samples,
        "train_rmsle": train_rmsle.mean(),
        "cv_rmsle": val_rmsle.mean(),
        "std_cv": val_rmsle.std(),
        "generalization_gap": (
            val_rmsle.mean() - train_rmsle.mean()
        ),
        "fit_time": scores["fit_time"].mean(),
    })

child_df = pd.DataFrame(child_results)

display(child_df.round(4))

,min_child_samples,train_rmsle,cv_rmsle,std_cv,generalization_gap,fit_time
0,20,1.1803,1.4708,0.0238,0.2906,1.0909
1,40,1.2121,1.4721,0.0268,0.2600,0.9142
2,80,1.2637,1.4710,0.0222,0.2073,0.7122


**Result:** CV RMSLE = **1.4708** (20), **1.4721** (40), **1.4710** (80). The 80-sample setting fits faster (~0.72s vs. ~1.06s) at essentially unchanged validation error; use **80** provisionally.

## 7. LightGBM: boosting iterations

Compare 75, 150, and 300 trees with `learning_rate=0.05`.

In [8]:
iteration_results = []

for n_trees in [75, 150, 300]:
    model = LGBMRegressor(
        n_estimators=n_trees,
        learning_rate=0.05,
        num_leaves=15,
        max_depth=6,
        min_child_samples=80,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        random_state=42,
        verbosity=-1,
        n_jobs=4,
    )

    scores = cross_validate(
        model,
        X_train,
        np.log1p(y_train),
        cv=cv,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
        n_jobs=1,
    )

    train_rmsle = -scores["train_score"]
    val_rmsle = -scores["test_score"]

    iteration_results.append({
        "n_estimators": n_trees,
        "train_rmsle": train_rmsle.mean(),
        "cv_rmsle": val_rmsle.mean(),
        "std_cv": val_rmsle.std(),
        "generalization_gap": (
            val_rmsle.mean() - train_rmsle.mean()
        ),
        "fit_time": scores["fit_time"].mean(),
    })

iteration_df = pd.DataFrame(iteration_results)

display(iteration_df.round(4))

,n_estimators,train_rmsle,cv_rmsle,std_cv,generalization_gap,fit_time
0,75,1.3595,1.4916,0.0196,0.1321,0.6601
1,150,1.2637,1.4710,0.0222,0.2073,0.7513
2,300,1.1534,1.4668,0.0232,0.3134,0.9776


**Result:** CV RMSLE falls from **1.4916** (75) to **1.4710** (150) and **1.4668** (300), with diminishing gains and a growing training–validation gap. Consider stopping behavior before adding more trees.

## 8. Early stopping on a temporary validation split

Allow up to 800 trees with 40-round patience; use only part of the training partition for the stopping check.

In [9]:
from lightgbm import LGBMRegressor, early_stopping
from sklearn.model_selection import train_test_split

X_fit, X_stop, y_fit, y_stop = train_test_split(
    X_train,
    np.log1p(y_train),
    test_size=0.2,
    random_state=42,
)

early_model = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.05,
    num_leaves=15,
    max_depth=6,
    min_child_samples=80,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1,
    n_jobs=4,
)

early_model.fit(
    X_fit,
    y_fit,
    eval_X=X_stop,
    eval_y=y_stop,
    eval_metric="rmse",
    callbacks=[
        early_stopping(stopping_rounds=40, verbose=False)
    ],
)

print("Best iteration:", early_model.best_iteration_)

Best iteration: 368


**Result:** Best iteration **368** on this single temporary split. Treat it as a candidate, not a final choice; validate fixed iteration counts across the five training folds.

## 9. Cross-validating the iteration choice

Compare 300, 368, 450, and 600 trees using identical folds and other settings.

In [10]:
iteration_cv_results = []

for n_trees in [300, 368, 450, 600]:
    model = LGBMRegressor(
        n_estimators=n_trees,
        learning_rate=0.05,
        num_leaves=15,
        max_depth=6,
        min_child_samples=80,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        random_state=42,
        verbosity=-1,
        n_jobs=4,
    )

    scores = cross_validate(
        model,
        X_train,
        np.log1p(y_train),
        cv=cv,
        scoring="neg_root_mean_squared_error",
        return_train_score=True,
        n_jobs=1,
    )

    train_rmsle = -scores["train_score"]
    val_rmsle = -scores["test_score"]

    iteration_cv_results.append({
        "n_estimators": n_trees,
        "train_rmsle": train_rmsle.mean(),
        "cv_rmsle": val_rmsle.mean(),
        "std_cv": val_rmsle.std(),
        "generalization_gap": (
            val_rmsle.mean() - train_rmsle.mean()
        ),
        "fit_time": scores["fit_time"].mean(),
    })

iteration_cv_df = pd.DataFrame(iteration_cv_results)

display(iteration_cv_df.round(4))

,n_estimators,train_rmsle,cv_rmsle,std_cv,generalization_gap,fit_time
0,300,1.1534,1.4668,0.0232,0.3134,1.1900
1,368,1.1166,1.4676,0.0235,0.3510,1.1430
2,450,1.0800,1.4703,0.0232,0.3904,1.2790
3,600,1.0230,1.4746,0.0224,0.4516,1.5766


**Result:** Mean CV RMSLE = **1.4668** (300), **1.4676** (368), **1.4703** (450), **1.4746** (600). Additional trees lower training error but increase validation error in this range.

## Selected candidate

| Model | Mean 5-fold CV RMSLE |
|---|---:|
| Decision Tree | 1.6458 |
| Random Forest (50 trees) | 1.5427 |
| Gradient Boosting | 1.5306 |
| **LightGBM (300 trees)** | **1.4668** |

**Decision:** Carry the 300-tree LightGBM into feature engineering. The reused CV folds support model selection, not an unbiased estimate of future performance.